In [1]:
import os
import json
import torch
import faiss
import bisect
import chromadb
import numpy as np
import polars as pl

from torch import nn

from math import ceil
from pathlib import Path
from collections import defaultdict
from datasets import load_from_disk
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from typing import Dict, Iterable, List, Optional, Any, Union, Literal, Tuple, Callable


In [2]:
class Tokenizer:
    def __init__(
        self,
        codes_parquet_fp: str,
        special_tokens: Optional[Iterable[str]] = None,
        force_special_ids: bool = True,  # pin [PAD]=0 etc.
    ):
        if special_tokens is None:
            special_tokens = ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]


        df_codes = pl.read_parquet(str(codes_parquet_fp), columns=["code"])
        base_codes = df_codes.get_column("code").to_list()
        seen = set()
        unique_codes = []
        for c in base_codes:
            if c not in seen:
                unique_codes.append(c)
                seen.add(c)

        vocab_list: List[str] = []
        special_tokens = list(special_tokens)

        if force_special_ids:
            for tok in ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]:
                if tok in special_tokens and tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)
                elif tok in special_tokens and tok in seen:

                    vocab_list.append(tok)
            for tok in special_tokens:
                if tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)
        else:
            for tok in special_tokens:
                if tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)


        vocab_list.extend(unique_codes)


        self.id2code: List[str] = vocab_list
        self.code2id: Dict[str, int] = {tok: idx for idx, tok in enumerate(vocab_list)}
        self.vocab_size: int = len(self.id2code)

        self.pad_token  = "[PAD]" if "[PAD]" in self.code2id else None
        self.mask_token = "[MASK]" if "[MASK]" in self.code2id else None
        self.cls_token  = "[CLS]" if "[CLS]" in self.code2id else None
        self.unk_token  = "[UNK]" if "[UNK]" in self.code2id else None

        self.pad_id  = self.code2id[self.pad_token]  if self.pad_token  else 0
        self.mask_id = self.code2id[self.mask_token] if self.mask_token else None
        self.cls_id  = self.code2id[self.cls_token]  if self.cls_token  else None
        self.unk_id  = self.code2id[self.unk_token]  if self.unk_token  else None


        type_set = set()
        for tok in self.id2code:
            prefix = tok.split("//", 1)[0]
            type_set.add(prefix)

        types_sorted = sorted(t for t in type_set if t not in ("[PAD]",))
        self.type2id: Dict[str, int] = {"[PAD]": 0}
        next_id = 1
        for sp in ["[MASK]", "[CLS]", "[UNK]"]:
            if sp in type_set:
                self.type2id[sp] = next_id; next_id += 1
        for t in types_sorted:
            if t not in self.type2id:
                self.type2id[t] = next_id
                next_id += 1

        self._code2id_df = pl.DataFrame({"code": self.id2code,
                                         "input_id": list(range(self.vocab_size))}) \
                               .with_columns(pl.col("code").cast(pl.Categorical))
        self._type2id_df = pl.DataFrame({"code_type": list(self.type2id.keys()),
                                         "type_id":   list(self.type2id.values())}) \
                               .with_columns(pl.col("code_type").cast(pl.Categorical))


    def encode(self, codes: Iterable[str]) -> List[int]:
        get = self.code2id.get
        if self.unk_id is not None:
            fallback = self.unk_id
        else:
            fallback = self.pad_id if self.pad_id is not None else 0
        return [get(c, fallback) for c in codes]

    def decode(self, ids: Iterable[int]) -> List[str]:
        out = []
        for i in ids:
            if 0 <= i < self.vocab_size:
                out.append(self.id2code[i])
            else:
                out.append(self.unk_token or "[UNK]")
        return out

    def save(self, path: str) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        obj = {
            "id2code": self.id2code,
            "special_tokens": [t for t in ["[PAD]", "[MASK]", "[CLS]", "[UNK]"] if t in self.code2id],
            "type2id": self.type2id,
            "pad_id": self.pad_id,
            "mask_id": self.mask_id,
            "cls_id": self.cls_id,
            "unk_id": self.unk_id,
        }
        with open(path, "w") as f:
            json.dump(obj, f, indent=2)

    @classmethod
    def load(cls, path: str) -> "Tokenizer":
        path = Path(path)
        with open(path) as f:
            obj = json.load(f)

        tok = cls.__new__(cls) 

        tok.id2code = obj["id2code"]
        tok.code2id = {tok_: i for i, tok_ in enumerate(tok.id2code)}
        tok.vocab_size = len(tok.id2code)

        tok.special_tokens = obj.get("special_tokens", [])
        tok.type2id = obj.get("type2id", {})

        tok.pad_token  = "[PAD]" if "[PAD]" in tok.code2id else None
        tok.mask_token = "[MASK]" if "[MASK]" in tok.code2id else None
        tok.cls_token  = "[CLS]" if "[CLS]" in tok.code2id else None
        tok.unk_token  = "[UNK]" if "[UNK]" in tok.code2id else None

        tok.pad_id  = obj.get("pad_id", tok.code2id.get("[PAD]", 0))
        tok.mask_id = obj.get("mask_id", tok.code2id.get("[MASK]")) if "[MASK]" in tok.code2id else None
        tok.cls_id  = obj.get("cls_id", tok.code2id.get("[CLS]"))   if "[CLS]" in tok.code2id else None
        tok.unk_id  = obj.get("unk_id", tok.code2id.get("[UNK]"))   if "[UNK]" in tok.code2id else None

        # Rebuild Polars lookup frames
        tok._code2id_df = pl.DataFrame({"code": tok.id2code,
                                        "input_id": list(range(tok.vocab_size))}) \
                              .with_columns(pl.col("code").cast(pl.Categorical))
        tok._type2id_df = pl.DataFrame({"code_type": list(tok.type2id.keys()),
                                        "type_id":   list(tok.type2id.values())}) \
                              .with_columns(pl.col("code_type").cast(pl.Categorical))
        return tok

    @property
    def code2id_df(self) -> pl.DataFrame:
        return self._code2id_df

    @property
    def type2id_df(self) -> pl.DataFrame:
        return self._type2id_df

In [3]:
class SequencesGenerator:
    def __init__(
        self,
        tokenizer_path: str,
        chunk_length: int = 1024,
        overlap: int = 128,
        return_numeric: bool = False,
        return_text: bool = False,
        return_time: bool = False,
        return_ids: bool = False,
    ):

        self.tokenizer = Tokenizer.load(tokenizer_path)
        self.chunk_length = chunk_length
        self.overlap = overlap
        self.return_numeric = return_numeric
        self.return_text = return_text
        self.return_time = return_time
        self.return_ids = return_ids

    def encode_sequence(
        self,
        timeline: pl.DataFrame,
        max_length: Optional[int] = None,
        pad_to_max: bool = False,
        truncation: Literal["head", "tail"] = "tail",
        add_cls: bool = False,
    ) -> Dict[str, Union[List[int], List[float], List[str]]]:
        """
        Vectorized build of:
          input_ids, attention_mask, visit_ids, stage_ids, type_ids
          + optional numeric/text streams (+ masks)
        """
        df = timeline
        
        if "seq_id" in df.columns:

            uniq = df.select(pl.col("seq_id")).unique(maintain_order=True)
            uniq = uniq.with_row_count(name="visit_ids_raw")  # 0..K-1
            df = df.join(uniq, on="seq_id", how="left").with_columns(
                (pl.col("visit_ids_raw") ).alias("visit_id").fill_null(0)
            ).drop("visit_ids_raw")
        else:
            df = df.with_columns(pl.lit(0).alias("visit_id"))

        stage_cols = ["out_id", "er_id", "hadm_id", "icustay_id"]
        present_stages = [c for c in stage_cols if c in df.columns]
        if present_stages:

            expr = pl.lit(0)
            for i, col in enumerate(present_stages, start=1):
                expr = pl.when(expr.eq(0) & pl.col(col).is_not_null()).then(i).otherwise(expr)
            df = df.with_columns(expr.alias("stage_id"))
        else:
            df = df.with_columns(pl.lit(0).alias("stage_id"))

        df = df.join(
            self.tokenizer.type2id_df,
            on=pl.col("code_type").cast(pl.Categorical),
            how="left",
        ).with_columns(pl.col("type_id").fill_null(0))

        df = df.join(
            self.tokenizer.code2id_df,
            on=pl.col("code").cast(pl.Categorical),
            how="left",
        )


        unk_id = self.tokenizer.unk_id if self.tokenizer.unk_id is not None else self.tokenizer.pad_id or 0
        df = df.with_columns(pl.col("input_id").fill_null(unk_id))

        df = df.with_columns(
            pl.when(pl.col("code").str.starts_with("TIME-GAP//"))
              .then(0)
              .otherwise(pl.col("visit_id"))
              .alias("visit_id"),
            pl.when(pl.col("code").str.starts_with("TIME-GAP//"))
              .then(0)
              .otherwise(pl.col("stage_id"))
              .alias("stage_id"),
        )


        if add_cls and (self.tokenizer.cls_token is not None):
            cls_row = {
                "code": self.tokenizer.cls_token,
                "code_type": "[CLS]",
                "visit_id": 0,
                "stage_id": 0,
                "type_id": self.tokenizer.type2id.get("[CLS]", 0),
                "input_id": self.tokenizer.cls_id,
            }

            if self.return_numeric:
                cls_row["numeric_value"] = None
            if self.return_text:
                cls_row["text_value"] = None

            df = pl.concat([pl.DataFrame([cls_row]), df], how="vertical_relaxed")


        input_ids = df.get_column("input_id").cast(pl.Int64).to_list()
        type_ids = df.get_column("type_id").cast(pl.Int64).to_list()
        visit_ids = df.get_column("visit_id").cast(pl.Int64).to_list()
        stage_ids = df.get_column("stage_id").cast(pl.Int64).to_list()
        attention_mask = [1] * len(input_ids)


        value_payload = self._build_value_streams(
            df=df,
            max_length=max_length,
            pad_to_max=pad_to_max,
            truncation=truncation,
        )

        # --- truncate/pad core streams in one go ---
        input_ids      = self._truncate(input_ids,      max_length, truncation)
        type_ids       = self._truncate(type_ids,       max_length, truncation)
        visit_ids      = self._truncate(visit_ids,      max_length, truncation)
        stage_ids      = self._truncate(stage_ids,      max_length, truncation)
        attention_mask = [1] * len(input_ids)

        if pad_to_max and max_length is not None and len(input_ids) < max_length:
            pad_len = max_length - len(input_ids)
            pad_id = self.tokenizer.pad_id if self.tokenizer.pad_id is not None else 0
            input_ids      = input_ids + [pad_id] * pad_len
            type_ids       = type_ids + [0] * pad_len
            visit_ids      = visit_ids + [0] * pad_len
            stage_ids      = stage_ids + [0] * pad_len
            attention_mask = attention_mask + [0] * pad_len

        out = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "visit_ids": visit_ids,
            "stage_ids": stage_ids,
            "type_ids": type_ids,
        }
        out.update(value_payload)
        return out

    def get_overlapped_chunks(
        self,
        timeline: Dict[str, Iterable],
        chunk_length: Optional[int] = None,
        overlap: Optional[int] = None,
        add_cls_per_chunk: bool = True,
    ) -> List[Dict[str, List[Any]]]:
        """
        Sliding-window chunking with optional [CLS] per chunk and padding.
        """
        if chunk_length is None or overlap is None:
            chunk_length = self.chunk_length
            overlap = self.overlap

        fields = ["input_ids", "attention_mask", "visit_ids", "stage_ids", "type_ids"]
        for extra in ("numeric_values", "numeric_mask", "text_values", "text_mask", 'time_diff'):
            if extra in timeline and extra not in fields:
                fields.append(extra)

        n = len(timeline["input_ids"])
        payload = chunk_length - (1 if add_cls_per_chunk else 0)
        step = max(1, payload - overlap)

        num_chunks = 1 if n <= payload else ceil((n - payload) / step) + 1
        starts = [i * step for i in range(num_chunks)]

        cls_defaults = {
            "input_ids": self.tokenizer.cls_id if self.tokenizer.cls_id is not None else (self.tokenizer.pad_id or 0),
            "attention_mask": 1,
            "visit_ids": 0,
            "stage_ids": 0,
            "type_ids": self.tokenizer.type2id.get("[CLS]", 0),
            "numeric_values": 0.0,
            "numeric_mask": 0,
            "text_values": "",
            "text_mask": 0,
            "time_diff":0.0}

        chunks = []
        for start in starts:
            end = min(n, start + payload)
            sliced = {k: list(timeline[k][start:end]) for k in fields if k in timeline}

            if "attention_mask" in sliced:
                sliced["attention_mask"] = [1] * len(sliced["input_ids"])

            if add_cls_per_chunk:
                for k in list(sliced.keys()):
                    sliced[k] = [cls_defaults[k]] + sliced[k]

            cur_len = len(sliced["input_ids"])
            if cur_len < chunk_length:
                pad_len = chunk_length - cur_len
                for k in list(sliced.keys()):
                    sliced[k] = self._pad_list(sliced[k], pad_len, 0)

            chunks.append(sliced)
        return chunks


    def _build_value_streams(
        self,
        df: pl.DataFrame,
        max_length: Optional[int],
        pad_to_max: bool,
        truncation: Literal["head", "tail"],
    ) -> Dict[str, List[Any]]:
        out: Dict[str, List[Any]] = {}
        # Numeric stream
        if self.return_numeric:
            if "numeric_value" in df.columns:
                vals = df.get_column("numeric_value").to_list()
            else:
                vals = [None] * df.height
            num_mask = [1 if (v is not None) else 0 for v in vals]
            vals = [0.0 if v is None else float(v) for v in vals]

            vals = self._truncate(vals, max_length, truncation)
            num_mask = self._truncate(num_mask, max_length, truncation)

            if pad_to_max and max_length is not None and len(vals) < max_length:
                pad_len = max_length - len(vals)
                vals += [0.0] * pad_len
                num_mask += [0] * pad_len

            out["numeric_values"] = vals
            out["numeric_mask"] = num_mask

        # Text stream
        if self.return_text:
            if "text_value" in df.columns:
                txt = df.get_column("text_value").to_list()
            else:
                txt = [None] * df.height
            txt = [("" if (t is None or str(t) == "___") else str(t)) for t in txt]
            txt_mask = [1 if (t != "") else 0 for t in txt]

            txt = self._truncate(txt, max_length, truncation)
            txt_mask = self._truncate(txt_mask, max_length, truncation)

            if pad_to_max and max_length is not None and len(txt) < max_length:
                pad_len = max_length - len(txt)
                txt += [""] * pad_len
                txt_mask += [0] * pad_len

            out["text_values"] = txt
            out["text_mask"] = txt_mask

            
        if self.return_time:
            if "time_diff" in df.columns:
                df = df.with_columns(pl.col(['time_diff'])).fill_null(0.0)
                time_diff = df.get_column("time_diff").to_list()
                time_diff = self._scale_time_deltas(time_diff)
                time_stamp = df.get_column("time").to_list()
            else:
                time_diff = [None] * df.height
                time_stamp = [None] * df.height


            if pad_to_max and max_length is not None and len(time_diff) < max_length:
                pad_len = max_length - len(time_diff)
                time_diff += [0] * pad_len
                time_stamp += [0] * pad_len


            out["time_diff"] = time_diff
            out["time_stamp"] = time_stamp
            
        if self.return_ids:
            if "seq_id" in df.columns:
                
                seq_id = df.get_column("seq_id").cast(pl.Int32).to_list()
                out_id = df.get_column("out_id").cast(pl.Int32).to_list()
                er_id =  df.get_column("er_id").cast(pl.Int32).to_list()
                hadm_id = df.get_column("hadm_id").cast(pl.Int32).to_list()
                icustay_id = df.get_column("hadm_id").cast(pl.Int32).to_list()
            else:
                seq_id = [None] * df.height
                out_id = [None] * df.height
                er_id =  [None] * df.height
                hadm_id = [None] * df.height
                icustay_id = [None] * df.height

            if pad_to_max and max_length is not None and len(time_diff) < max_length:
                pad_len = max_length - len(time_diff)
                seq_id += [0] * pad_len
                out_id += [0] * pad_len
                er_id += [0] * pad_len
                hadm_id += [0] * pad_len
                icustay_id += [0] * pad_len

            out["seq_id"] = seq_id
            out["out_id"] = out_id
            out["er_id"] = er_id
            out["hadm_id"] = hadm_id
            out["icustay_id"] = icustay_id

        return out
    
    def get_time_based_chunks(
        self,
        timeline: Dict[str, Iterable],
        window_hours: float,
        anchor_from_first_valid_time: bool = True,
        keep_prefix_tokens: bool = True,
    ) -> List[Dict[str, List[Any]]]:

        if "input_ids" not in timeline:
            raise ValueError("timeline must contain 'input_ids'")
        if "time_stamp" not in timeline:
            raise ValueError("timeline must contain 'time_stamp' for time-based chunking")

        n = len(timeline["input_ids"])
        if n == 0:
            return []

        fields = [k for k, v in timeline.items() if isinstance(v, (list, tuple))]
        time_stamps = list(timeline["time_stamp"])

        def _is_valid_time(x: Any) -> bool:
            return x is not None

        anchor_idx = None
        for i, ts in enumerate(time_stamps):
            if _is_valid_time(ts):
                anchor_idx = i
                break

        if anchor_idx is None:
            return [{k: list(v) for k, v in timeline.items() if isinstance(v, (list, tuple))}]

        t0 = time_stamps[anchor_idx]
        window_size = window_hours * 3600.0

        if window_size <= 0:
            raise ValueError("window_hours must be > 0")

        prefix_idx = list(range(anchor_idx)) if keep_prefix_tokens else []

        buckets: Dict[int, List[int]] = {}
        for i in range(anchor_idx, n):
            ts = time_stamps[i]

            if not _is_valid_time(ts):
                window_id = 0
            else:
                elapsed = (ts - t0).total_seconds()
                window_id = int(elapsed // window_size) if elapsed >= 0 else 0

            buckets.setdefault(window_id, []).append(i)

        chunks: List[Dict[str, List[Any]]] = []
        for chunk_idx, window_id in enumerate(sorted(buckets.keys())):
            idxs = (prefix_idx + buckets[window_id]) if chunk_idx == 0 else buckets[window_id]

            chunk = {}
            for k in fields:
                chunk[k] = [timeline[k][j] for j in idxs]

            chunks.append(chunk)

        final_chunks = []

        for chunk in chunks:
            sub_chunks = self.get_overlapped_chunks(
                timeline=chunk,
                chunk_length=self.chunk_length,
                overlap=0,
                add_cls_per_chunk=True
            )
            final_chunks.extend(sub_chunks)

        return final_chunks
    
    def get_visit_level_chunks(
        self,
        timeline: Dict[str, Iterable],
        keep_prefix_tokens: bool = True,
    ) -> List[Dict[str, List[Any]]]:

        if "input_ids" not in timeline:
            raise ValueError("timeline must contain 'input_ids'")
        if "seq_id" not in timeline:
            raise ValueError("timeline must contain 'seq_id' for visit-level chunking")

        n = len(timeline["input_ids"])
        if n == 0:
            return []

        fields = [k for k, v in timeline.items() if isinstance(v, (list, tuple))]
        seq_ids = list(timeline["seq_id"])

        def _is_valid_visit(x: Any) -> bool:
            return x is not None and x != 0

        # find first actual visit event
        first_visit_idx = None
        for i, sid in enumerate(seq_ids):
            if _is_valid_visit(sid):
                first_visit_idx = i
                break

        # if no valid seq_id exists, return overlapped chunks on full sequence
        if first_visit_idx is None:
            return self.get_overlapped_chunks(
                timeline={k: list(v) for k, v in timeline.items() if isinstance(v, (list, tuple))},
                chunk_length=self.chunk_length,
                overlap=0,
                add_cls_per_chunk=True,
            )

        prefix_idx = list(range(first_visit_idx)) if keep_prefix_tokens else []

        buckets: Dict[Any, List[int]] = {}
        visit_order: List[Any] = []

        for i in range(first_visit_idx, n):
            sid = seq_ids[i]
            if not _is_valid_visit(sid):
                continue

            if sid not in buckets:
                buckets[sid] = []
                visit_order.append(sid)
            buckets[sid].append(i)

        raw_chunks: List[Dict[str, List[Any]]] = []
        for chunk_idx, sid in enumerate(visit_order):
            idxs = (prefix_idx + buckets[sid]) if chunk_idx == 0 else buckets[sid]

            chunk = {}
            for k in fields:
                chunk[k] = [timeline[k][j] for j in idxs]

            raw_chunks.append(chunk)

        final_chunks: List[Dict[str, List[Any]]] = []
        for chunk in raw_chunks:
            sub_chunks = self.get_overlapped_chunks(
                timeline=chunk,
                chunk_length=self.chunk_length,
                overlap=0,
                add_cls_per_chunk=True,
            )
            final_chunks.extend(sub_chunks)

        return final_chunks

    
    def get_care_stage_level_chunks(
        self,
        timeline: Dict[str, Iterable],
        keep_prefix_tokens: bool = True,
    ) -> List[Dict[str, List[Any]]]:

        if "input_ids" not in timeline:
            raise ValueError("timeline must contain 'input_ids'")
        if "seq_id" not in timeline:
            raise ValueError("timeline must contain 'seq_id' for care-stage chunking")

        n = len(timeline["input_ids"])
        if n == 0:
            return []

        fields = [k for k, v in timeline.items() if isinstance(v, (list, tuple))]

        seq_ids = list(timeline["seq_id"]) if "seq_id" in timeline else [None] * n
        er_ids = list(timeline["er_id"]) if "er_id" in timeline else [None] * n
        out_ids = list(timeline["out_id"]) if "out_id" in timeline else [None] * n
        hadm_ids = list(timeline["hadm_id"]) if "hadm_id" in timeline else [None] * n
        icu_ids = list(timeline["icustay_id"]) if "icustay_id" in timeline else [None] * n

        def _valid(x: Any) -> bool:
            return x is not None and x != 0

        # make hadm mutually exclusive with icu at the row level
        hadm_ids = [
            None if _valid(icu) else hadm
            for hadm, icu in zip(hadm_ids, icu_ids)
        ]

        # per-row stage priority
        def _stage_key(i: int):
            if _valid(icu_ids[i]):
                return ("icu", icu_ids[i])
            elif _valid(hadm_ids[i]):
                return ("hadm", hadm_ids[i])
            elif _valid(er_ids[i]):
                return ("er", er_ids[i])
            elif _valid(out_ids[i]):
                return ("out", out_ids[i])
            return None

        # first row that belongs to a visit
        first_visit_idx = None
        for i in range(n):
            if _valid(seq_ids[i]):
                first_visit_idx = i
                break

        if first_visit_idx is None:
            return self.get_overlapped_chunks(
                timeline={k: list(v) for k, v in timeline.items() if isinstance(v, (list, tuple))},
                chunk_length=self.chunk_length,
                overlap=0,
                add_cls_per_chunk=True,
            )

        prefix_idx = list(range(first_visit_idx)) if keep_prefix_tokens else []

        # group by (seq_id, stage_type, stage_id), preserving first-seen order
        buckets: Dict[Any, List[int]] = {}
        chunk_order: List[Any] = []

        for i in range(first_visit_idx, n):
            if not _valid(seq_ids[i]):
                continue

            stage = _stage_key(i)
            if stage is None:
                continue

            key = (seq_ids[i], stage[0], stage[1])

            if key not in buckets:
                buckets[key] = []
                chunk_order.append(key)
            buckets[key].append(i)

        final_chunks: List[Dict[str, List[Any]]] = []

        for chunk_idx, key in enumerate(chunk_order):
            idxs = (prefix_idx + buckets[key]) if chunk_idx == 0 else buckets[key]

            chunk = {}
            for k in fields:
                chunk[k] = [timeline[k][j] for j in idxs]

            sub_chunks = self.get_overlapped_chunks(
                timeline=chunk,
                chunk_length=self.chunk_length,
                overlap=0,
                add_cls_per_chunk=True,
            )

#             for sub_chunk in sub_chunks:
#                 sub_chunk["source_seq_id"] = key[0]
#                 sub_chunk["source_stage_type"] = key[1]
#                 sub_chunk["source_stage_id"] = key[2]

            final_chunks.extend(sub_chunks)

        return final_chunks
    
    def _scale_time_deltas(self, deltas_list):
        deltas = np.asarray(deltas_list, dtype=float)
        compressed = np.log1p(deltas)              
        scaled = compressed / np.log(5328.93125)         
        return scaled.tolist()

    @staticmethod
    def _truncate(seq: List[Any], max_length: Optional[int], truncation: str) -> List[Any]:
        if max_length is None or len(seq) <= max_length:
            return seq
        return seq[-max_length:] if truncation == "head" else seq[:max_length]

    @staticmethod
    def _pad_list(lst: List[Any], pad_len: int, pad_value: Any) -> List[Any]:
        if pad_len <= 0:
            return lst
        return lst + [pad_value] * pad_len

In [4]:
limits = {
    'within24_query': {512:  ['w24_start_512',  'w24_end_512' ],
                       1024: ['w24_start_1024', 'w24_end_1024'],
                       1536: ['w24_start_1536', 'w24_end_1536'],
                       2048: ['w24_start_2048', 'w24_end_2048'],
                      },
    
    'within24_hist_icu': {512: ['wStay_min', 'w24_start_512' ],
                         1024: ['wStay_min', 'w24_start_1024'],
                         1536: ['wStay_min', 'w24_start_1536'],
                         2048: ['wStay_min', 'w24_start_2048'],
                      },
    
    'within24_hist_full': {512: [ 0, 'w24_start_512' ],
                          1024: [ 0, 'w24_start_1024'],
                          1536: [ 0, 'w24_start_1536'],
                          2048: [ 0, 'w24_start_2048'],
                          },

    
    'within48_query': {512:  ['w48_start_512',  'w48_end_512' ],
                       1024: ['w48_start_1024', 'w48_end_1024'],
                       1536: ['w48_start_1536', 'w48_end_1536'],
                       2048: ['w48_start_2048', 'w48_end_2048'],
                      },

    'within48_hist_icu': {512: ['wStay_min', 'w48_start_512' ],
                         1024: ['wStay_min', 'w48_start_1024'],
                         1536: ['wStay_min', 'w48_start_1536'],
                         2048: ['wStay_min', 'w48_start_2048'],
                      },
    
    'within48_hist_full': {512: [ 0, 'w48_start_512' ],
                          1024: [ 0, 'w48_start_1024'],
                          1536: [ 0, 'w48_start_1536'],
                          2048: [ 0, 'w48_start_2048'],
                          },
    
    'within_stay_query': {512:  ['wStay_start_512',  'wStay_end_512' ],
                          1024: ['wStay_start_1024', 'wStay_end_1024'],
                          1536: ['wStay_start_1536', 'wStay_end_1536'],
                          2048: ['wStay_start_2048', 'wStay_end_2048'],
                         },
    
    'within_stay_hist_icu': {512:  ['wStay_min', 'wStay_start_512' ],
                             1024: ['wStay_min', 'wStay_start_1024'],
                             1536: ['wStay_min', 'wStay_start_1536'],
                             2048: ['wStay_min', 'wStay_start_2048'],
                            },
    
    'within_stay_hist_full': {512:  [ 0, 'wStay_start_512' ],
                              1024: [ 0, 'wStay_start_1024'],
                              1536: [ 0, 'wStay_start_1536'],
                              2048: [ 0, 'wStay_start_2048'],
                             },
    }

In [5]:
# Updated
class EvalCollator:
    def __init__(self) -> None:
        self.meta_keys = {"subject_id"}#, "sample_idx", "hadm_id", "icustay_id"}

    def __call__(self, batch: List[Union[Dict, List[Dict]]]) -> Dict[str, torch.Tensor]:
        chunks = self._flatten(batch)
        out = self._stack(chunks)

        if "numeric_values" in out:
            vals = out["numeric_values"].float()
            finite_mask = torch.isfinite(vals)

            if "numeric_mask" in out:
                mask = out["numeric_mask"].bool() & finite_mask
            else:
                mask = finite_mask

            vals = torch.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)

            out["numeric_values"] = vals
            out["numeric_mask"] = mask
        if "time_diff" in out:
            t = out["time_diff"].float()
            t = torch.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0)
            out["time_diff"] = t

        return out

    def _flatten(self, batch) -> List[Dict]:
        out: List[Dict] = []
        for item in batch:
            if isinstance(item, dict):
                out.append(item)
            elif isinstance(item, (list, tuple)):
                out.extend(item)
            else:
                raise TypeError(f"Unexpected item type: {type(item)}")
        if not out:
            raise ValueError("Empty batch after normalization.")
        return out

    def _stack(self, chunks: List[Dict]) -> Dict[str, torch.Tensor]:
        keys = list(chunks[0].keys())
        out = {}

        for k in keys:
            if k in ("text_values",):
                continue

            if k in self.meta_keys:
                vals = []
                for c in chunks:
                    v = c.get(k, None)
                    if v is None:
                        vals.append(-1)
                    else:
                        vals.append(int(v))
                out[k] = torch.tensor(vals, dtype=torch.long)
                continue

            seq_list = []
            for c in chunks:
                v = c.get(k, None)
                if isinstance(v, list):
                    v = [0 if x is None else x for x in v]
                elif v is None:
                    v = 0
                seq_list.append(torch.as_tensor(v))
            out[k] = torch.stack(seq_list, 0)

        return out

In [6]:
import json
import torch
import numpy as np
import polars as pl

import torch.nn as nn
from tqdm import tqdm
from datasets import load_from_disk
from collections import defaultdict
from transformers import RoFormerModel
from typing import List, Dict, Union, Optional
from chromadb.api.types import Documents, Embeddings, EmbeddingFunction

In [7]:
class EHREmbedder(nn.Module):
    def __init__(
        self,
        config,
        backbone,
        ckpt_path: Optional[str] = None,
        dropout: float = 0.1,
        pooling: str = "cls",
#         normalize: bool = False,
        use_numeric: bool = False, # not to be used, but implemenetd for future research, Keep always False
        use_time: bool = False, # not to be used, but implemenetd for future research, Keep always False
        device: Optional[str] = None,
        dtype: torch.dtype = torch.float32,
    ):
        super().__init__()
        self.config = config
        self.pooling = pooling
#         self.normalize = normalize
        self.dtype_ = dtype
        self.device_ = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.backbone = backbone(config)
        self.use_time = use_time
        self.use_numeric = use_numeric
        
        if self.use_time or self.use_numeric:
            raise ValueError("not to be used, but implemenetd for future research, Keep always False")
            
        rope_model_types = {"modernbert", "roformer"}
        model_type = getattr(config, "model_type", "").lower()
        is_rope = model_type in rope_model_types
        
        self.ehr_embeddings = EHREmbeddings(
            vocab_size=config.vocab_size,
            embedding_size=getattr(config, "hidden_size", None) or getattr(config, "embedding_size", None),
            pad_token_id=config.pad_token_id,
            type_vocab_size=config.type_vocab_size,
            visit_vocab_size=config.visit_vocab_size,
            stage_vocab_size=config.stage_vocab_size,
            dropout=dropout,
            use_position_embeddings=not is_rope,
            max_position_embeddings=(getattr(config, "max_position_embeddings", 0) if not is_rope else 0),
            use_time=self.use_time,
            time_in_features=1,
            time_out_features=16,
            use_numeric=self.use_numeric,
        )

        if ckpt_path:
            self.load_pretrained_weights(ckpt_path)

        self.eval().to(self.device_, dtype=self.dtype_)

    @torch.no_grad()
    def encode(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        type_ids: torch.Tensor,
        visit_ids: torch.Tensor,
        stage_ids: torch.Tensor,
        time_feats: Optional[torch.Tensor] = None,
        numeric_values: Optional[torch.Tensor] = None,
        numeric_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if (time_feats is not None) or (numeric_values is not None) or (numeric_mask is not None):
            raise ValueError(
                "EHREmbedder.encode: time/numeric tensors were passed but are disabled. "
                "Pass None for time_feats/numeric_values/numeric_mask.")
        
        inputs_embeds = self.ehr_embeddings.encode(
            input_ids=input_ids.to(self.device_),
            type_ids=type_ids.to(self.device_),
            visit_ids=visit_ids.to(self.device_),
            stage_ids=stage_ids.to(self.device_),
            time_feats=(time_feats.to(self.device_) if time_feats is not None else None),
            numeric_values=(numeric_values.to(self.device_) if numeric_values is not None else None),
            numeric_mask=(numeric_mask.to(self.device_) if numeric_mask is not None else None)).to(self.dtype_)

        outputs = self.backbone(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask.to(self.device_),
            output_hidden_states=False,
            return_dict=True,
        )
        last_hidden = outputs.last_hidden_state  

        if self.pooling == "cls":
            vec = last_hidden[:, 0, :]
        elif self.pooling == "mean":
            mask = attention_mask.unsqueeze(-1).to(device=last_hidden.device,dtype=last_hidden.dtype) 
            vec = (last_hidden * mask).sum(dim=1) / (mask.sum(dim=1).clamp(min=1.0))
        else:
            raise ValueError(f"Unsupported pooling='{self.pooling}' (use 'cls' or 'mean')")

#         if self.normalize:
#             vec = nn.functional.normalize(vec, p=2, dim=-1)
        return vec

    def load_pretrained_weights(self, ckpt_path: str) -> None:
        sd_obj = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        sd = sd_obj["state_dict"] if isinstance(sd_obj, dict) and "state_dict" in sd_obj else sd_obj

        mt = getattr(self.config, "model_type", "").lower()
        prefix_map = {
            "bert":       "backbone.bert.",
            "roberta":    "backbone.roberta.",
            "longformer": "backbone.longformer.",
            "modernbert": "backbone.model.",
            "roformer":   "backbone.roformer.",
            "big_bird":   "backbone.bert.",
            "mamba":      "backbone.backbone.",
            "mamba2":     "backbone.backbone."}
        backbone_prefix = prefix_map.get(mt, None)

        DROP_PREFIXES = ["backbone.cls.", "top_1_train.", "top_1_val.", "backbone.lm_head.", 
                         "classifier.", "cls.", "lm_head.", "score."]
        remapped = {}
        for k, v in sd.items():
            if any(k.startswith(dp) for dp in DROP_PREFIXES):
                continue

            if k.startswith("ehr_embeddings."):
                new_k = k
            elif backbone_prefix and k.startswith(backbone_prefix):
                new_k = "backbone." + k[len(backbone_prefix):]
            elif k.startswith("backbone."):
                new_k = k
            else:
                new_k = k

            remapped[new_k] = v
        missing, unexpected = self.load_state_dict(remapped, strict=False)
        print("Embedder weights loaded.")
        print("missing keys:", missing)
        print("unexpected keys:", unexpected)

In [8]:

import os
import yaml
import torch
import wandb
import torch.nn as nn
import torch.distributed as dist

from typing import Callable
from transformers import BertConfig
from torchmetrics.classification import BinaryAUROC, BinaryAveragePrecision
from transformers import CONFIG_MAPPING, MODEL_FOR_MASKED_LM_MAPPING, MODEL_MAPPING, MODEL_FOR_CAUSAL_LM_MAPPING




BERT_VARIANTS = {
    "bert": {},
    "medbert": dict(
        hidden_size=192,
        intermediate_size=64,
        num_attention_heads=6,
        num_hidden_layers=6,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1,
    ),

    "cehrbert": dict(
        hidden_size=128,
        intermediate_size=2048,
        num_hidden_layers=12,
        num_attention_heads=8,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1,
    ),

    "behrt": dict(
        hidden_size=288,
        intermediate_size=512,
        num_attention_heads=12,
        num_hidden_layers=6,
        hidden_dropout_prob=0.1,
        attention_probs_dropout_prob=0.1,
    ),

    "hibehrt": dict(
        hidden_size=150,
        intermediate_size=108,
        num_attention_heads=6,
        num_hidden_layers=4,
        hidden_dropout_prob=0.2,
        attention_probs_dropout_prob=0.3,
    ),
}



def get_config_and_model_cls(model_type: str, mode: str = "mlm", variant: str = None):
    assert mode in ["mlm", "eval", "causal"]

    if model_type not in CONFIG_MAPPING:
        raise ValueError(f"Unknown model_type: {model_type}")

    config_cls = CONFIG_MAPPING[model_type]

    if mode == "mlm":
        model_cls = MODEL_FOR_MASKED_LM_MAPPING[config_cls]
    elif mode == "eval":
        model_cls = MODEL_MAPPING[config_cls]
    else:
        model_cls = MODEL_FOR_CAUSAL_LM_MAPPING[config_cls]

    variant_kwargs = {}
    if variant is not None and issubclass(config_cls, BertConfig):
        variant_kwargs = BERT_VARIANTS.get(variant, {})
        if variant not in BERT_VARIANTS:
            raise ValueError(f"Unknown BERT variant: {variant}")

    def build_config(**kwargs):
        return config_cls(**variant_kwargs, **kwargs)

    return build_config, model_cls


def fix_roberta_longformer_max_pos(cfg):

    model_type = getattr(cfg, "model_type", "").lower()

    if model_type == "roberta":
        if cfg.max_position_embeddings == 512:
            cfg.max_position_embeddings = 513

    elif model_type == "longformer":
        if cfg.max_position_embeddings == 512:
            cfg.max_position_embeddings = 4097

    return cfg



def load_config_with_env(path):
    # read file
    with open(path, "r") as f:
        raw_text = f.read()
    expanded = os.path.expandvars(raw_text)
    
    return yaml.safe_load(expanded)


class Time2Vec(nn.Module):

    def __init__(
        self,
        in_features: int = 1,
        out_features: int = 16,
        periodic_activation: Callable = torch.sin,
    ):
        super().__init__()
        assert out_features >= 1, "out_features must be >= 1"

        self.in_features = in_features
        self.out_features = out_features
        self.periodic_activation = periodic_activation

        self.W = nn.Parameter(torch.randn(in_features, out_features - 1))
        self.b = nn.Parameter(torch.randn(out_features - 1))

        self.W0 = nn.Parameter(torch.randn(in_features))
        self.b0 = nn.Parameter(torch.randn(1))

    def forward(self, tau: torch.Tensor) -> torch.Tensor:

        v1 = self.periodic_activation(tau @ self.W + self.b)
        v2 = (tau @ self.W0).unsqueeze(-1) + self.b0

        return torch.cat([v2, v1], dim=-1)
    

def get_rank():
    if not dist.is_available() or not dist.is_initialized():
        return 0
    return dist.get_rank()




def get_bootstrap_ci(
    y_true: torch.Tensor,
    y_score: torch.Tensor,
    num_iter: int = 1000,
    alpha: float = 0.05,
    ndigits: int = 3,
):
    device = y_score.device

    y_true = y_true.detach().view(-1).to(device).long()
    y_score = y_score.detach().view(-1).to(device)

    auroc_point = BinaryAUROC().to(device)(y_score, y_true)
    auprc_point = BinaryAveragePrecision().to(device)(y_score, y_true)

    n = y_true.numel()
    auroc_samples = torch.empty(num_iter, device=device)
    auprc_samples = torch.empty(num_iter, device=device)

    for i in range(num_iter):
        idx = torch.randint(0, n, (n,), device=device)
        auroc_samples[i] = BinaryAUROC().to(device)(y_score[idx], y_true[idx])
        auprc_samples[i] = BinaryAveragePrecision().to(device)(y_score[idx], y_true[idx])

    # Percentile CI
    q_low = alpha / 2.0         # 2.5%
    q_high = 1.0 - alpha / 2.0  # 97.5%

    auroc_low = torch.quantile(auroc_samples, q_low)
    auroc_high = torch.quantile(auroc_samples, q_high)

    auprc_low = torch.quantile(auprc_samples, q_low)
    auprc_high = torch.quantile(auprc_samples, q_high)

    def _fmt(point, low, high):
        p = float(point.detach().cpu())
        l = float(low.detach().cpu())
        h = float(high.detach().cpu())
        return f"{round(p, ndigits)} ({round(l, ndigits)}, {round(h, ndigits)})"

    auroc_text = _fmt(auroc_point, auroc_low, auroc_high)
    auprc_text = _fmt(auprc_point, auprc_low, auprc_high)

    return auroc_text, auprc_text


def gather_1d_varlen_pl(module, x: torch.Tensor) -> torch.Tensor:
    x = x.detach().view(-1)

    if not getattr(module, "trainer", None) or module.trainer.world_size == 1:
        return x

    device = x.device
    local_len = torch.tensor([x.numel()], device=device, dtype=torch.long)

    all_lens = module.all_gather(local_len).view(-1) 
    max_len = int(all_lens.max().item())

    if x.numel() < max_len:
        pad = torch.zeros(max_len - x.numel(), device=device, dtype=x.dtype)
        x_pad = torch.cat([x, pad], dim=0)
    else:
        x_pad = x

    x_gather = module.all_gather(x_pad)

    chunks = []
    for r in range(x_gather.shape[0]):
        chunks.append(x_gather[r, : int(all_lens[r].item())])
    return torch.cat(chunks, dim=0)


def log_bootstrap_ci_text_percentile(
    module,
    y_true: torch.Tensor,
    y_score: torch.Tensor,
    prefix: str = "test",
    num_iter: int = 1000,
    alpha: float = 0.05,
    ndigits: int = 3,
):
    y_all = gather_1d_varlen_pl(module, y_true)
    s_all = gather_1d_varlen_pl(module, y_score)

    if not getattr(module, "trainer", None) or module.trainer.is_global_zero:
        auroc_ci_text, auprc_ci_text = get_bootstrap_ci(
            y_true=y_all,
            y_score=s_all,
            num_iter=num_iter,
            alpha=alpha,
            ndigits=ndigits,
        )

        # Use wandb.log directly for string-based CI values
        wandb.log({
            f"{prefix}_auroc_ci": auroc_ci_text,
            f"{prefix}_auprc_ci": auprc_ci_text,
        }, commit=False)

In [9]:

import os
import torch


import lightning as lt
import torch.nn as nn
import torch.nn.functional as F
from typing import Any, Dict, Optional

from torchmetrics.classification import Accuracy, BinaryAUROC, BinaryAveragePrecision

In [10]:
 class EHREmbeddings(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_size: int,
        pad_token_id: int = 0,
        type_vocab_size: int = 28,
        visit_vocab_size: int = 102,
        stage_vocab_size: int = 5,
        dropout: float = 0.1,
        use_position_embeddings: bool = False,
        max_position_embeddings: int = 0,
        use_time: bool = False,
        time_in_features: int = 1,
        time_out_features: int = 16,
        use_numeric: bool = False,
        numeric_hidden_size: int = 16,   
    ):
        super().__init__()

        self.tok_emb   = nn.Embedding(vocab_size,       embedding_size, padding_idx=pad_token_id)
        self.type_emb  = nn.Embedding(type_vocab_size,  embedding_size, padding_idx=pad_token_id)
        self.visit_emb = nn.Embedding(visit_vocab_size, embedding_size, padding_idx=pad_token_id)
        self.stage_emb = nn.Embedding(stage_vocab_size, embedding_size, padding_idx=pad_token_id)

        
        self.use_position_embeddings = use_position_embeddings
        if use_position_embeddings:
            if max_position_embeddings <= 0:
                raise ValueError("max_position_embeddings must be > 0 when use_position_embeddings=True")
            self.pos_emb = nn.Embedding(max_position_embeddings, embedding_size)
        else:
            self.pos_emb = None

        
        self.use_time = use_time
        if use_time:
            self.time2vec = Time2Vec(
                in_features=time_in_features,
                out_features=time_out_features,
                periodic_activation=torch.sin,
            )
            self.time_proj = nn.Linear(time_out_features, embedding_size)
        else:
            self.time2vec = None
            self.time_proj = None

        
        self.use_numeric = use_numeric
        if use_numeric:
            self.numeric_hidden_size = numeric_hidden_size
            
            self.num_proj1 = nn.Linear(1, numeric_hidden_size)
            self.num_proj2 = nn.Linear(numeric_hidden_size, embedding_size)
            self.num_act = nn.GELU()

            
            self.null_numeric = nn.Parameter(torch.zeros(embedding_size))
            nn.init.normal_(self.null_numeric, mean=0.0, std=0.02)

            nn.init.xavier_uniform_(self.num_proj1.weight)
            nn.init.zeros_(self.num_proj1.bias)
            nn.init.xavier_uniform_(self.num_proj2.weight)
            nn.init.zeros_(self.num_proj2.bias)
        else:
            self.num_proj1 = None
            self.num_proj2 = None
            self.num_act = None
            self.null_numeric = None

        self.norm = nn.LayerNorm(embedding_size)
        self.drop = nn.Dropout(dropout)

    def encode(
        self,
        input_ids,
        type_ids,
        visit_ids,
        stage_ids,
        time_feats=None,          
        numeric_values=None,     
        numeric_mask=None,        
    ):
        
        x = self.tok_emb(input_ids.long())
        x = x + self.type_emb(type_ids.long())
        x = x + self.visit_emb(visit_ids.long())
        x = x + self.stage_emb(stage_ids.long())

        
        if self.pos_emb is not None:
            shape = input_ids.size()
            seqlen = shape[-1]

            position_ids = torch.arange(seqlen, device=input_ids.device)

            if input_ids.dim() == 2:        
                bsz = shape[0]
                position_ids = position_ids.unsqueeze(0).expand(bsz, seqlen)          
            elif input_ids.dim() == 3:        
                bsz, n = shape[0], shape[1]
                position_ids = position_ids.view(1, 1, seqlen).expand(bsz, n, seqlen) 
            else:
                raise ValueError(f"Unsupported input_ids.dim()={input_ids.dim()}")
            x = x + self.pos_emb(position_ids)

        
        if self.use_time:
            if time_feats is None:
                raise ValueError("time_feats must be provided when use_time=True")
            if time_feats.dim() == 2:
                time_feats = time_feats.unsqueeze(-1)
            elif time_feats.dim() != 3:
                raise ValueError(f"Unexpected time_feats.dim()={time_feats.dim()}, expected 2 or 3")
            t = self.time2vec(time_feats.float())   
            t = self.time_proj(t)                   
            x = x + t

        
        if self.use_numeric:
            if numeric_values is None or numeric_mask is None:
                raise ValueError("numeric_values and numeric_mask must be provided when use_numeric=True")

            
            v = numeric_values.float().unsqueeze(-1)       
            
            h = self.num_act(self.num_proj1(v))             
            num_emb = self.num_proj2(h)                     

            mask = numeric_mask.bool().unsqueeze(-1)        
            num_emb = torch.where(mask, num_emb, self.null_numeric.view(1, 1, -1))
            x = x + num_emb

        return self.drop(self.norm(x))

    def forward(self, input_ids=None, token_type_ids=None, inputs_embeds=None, **kwargs):
        if inputs_embeds is not None:
            return inputs_embeds
        x = self.tok_emb(input_ids.long())
        return self.drop(self.norm(x))

In [11]:
def _build_index(hf_dataset):
    sids = hf_dataset["subject_id"]
    index = defaultdict(list)
    for i, sid in enumerate(sids):
        index[sid].append(i)
    return index

def _to_list(self, v):
    if isinstance(v, np.ndarray):
        return v.tolist()
    if isinstance(v, list):
        return v
    return v

In [12]:
def build_faiss_index(ch_embs, metric: str = "l2"):

    # Accept either:
    #  - list[Tensor] each (1,D) or (D,)
    #  - Tensor of shape (N,D) or (D,) or (1,D)
    if torch.is_tensor(ch_embs):
        v = ch_embs.detach().cpu()
        if v.ndim == 1:
            xb = v.unsqueeze(0).numpy()
        elif v.ndim == 2:
            xb = v.numpy()
        else:
            raise ValueError(f"ch_embs tensor must be (D,) or (N,D). Got {tuple(v.shape)}")
    else:
        xb_list = []
        for t in ch_embs:
            if not torch.is_tensor(t):
                raise TypeError(f"Expected torch.Tensor embeddings, got {type(t)}")
            v = t.detach().cpu()
            if v.ndim == 2 and v.shape[0] == 1:
                v = v[0]
            elif v.ndim != 1:
                raise ValueError(f"Each embedding must be (D,) or (1,D). Got {tuple(v.shape)}")
            xb_list.append(v.numpy())
        xb = np.stack(xb_list, axis=0)

    xb = xb.astype(np.float32, copy=False)
    xb = np.ascontiguousarray(xb)  # <-- FIX

    dim = xb.shape[1]
    metric = metric.lower()

    if metric in ("cosine", "ip", "inner_product", "dot"):
        faiss.normalize_L2(xb)      # now safe
        index = faiss.IndexFlatIP(dim)
    elif metric in ("l2", "euclidean"):
        index = faiss.IndexFlatL2(dim)
    else:
        raise ValueError(f"Unknown metric={metric}. Use 'l2' or 'cosine'.")

    index.add(xb)
    return index, xb

In [13]:
def query_faiss(index, q_emb, top_k: int = 10, metric: str = "l2"):
    import numpy as np
    import faiss
    import torch

    if not torch.is_tensor(q_emb):
        raise TypeError(f"q_emb must be torch.Tensor, got {type(q_emb)}")

    xq = q_emb.detach().cpu()
    if xq.ndim == 1:
        xq = xq.unsqueeze(0)
    elif xq.ndim == 2:
        pass
    else:
        raise ValueError(f"q_emb must be (D,) or (B,D). Got {tuple(xq.shape)}")

    xq = xq.numpy().astype(np.float32, copy=False)
    xq = np.ascontiguousarray(xq)  # <-- FIX

    metric = metric.lower()
    if metric in ("cosine", "ip", "inner_product", "dot"):
#         faiss.normalize_L2(xq)
        scores, ids = index.search(xq, top_k)  # cosine sims (higher better)
        return scores, ids
    elif metric in ("l2", "euclidean"):
        dists, ids = index.search(xq, top_k)   # squared L2 (lower better)
        return dists, ids
    else:
        raise ValueError(f"Unknown metric={metric}. Use 'l2' or 'cosine'.")

In [14]:
def build_indices(data_idx_path:str,
                  hf_dataset_path: str,
                  tokenizer_path: str,
                  save_path:str,
                  main_window_q: str,
                  main_window_h: str,
                  limits_dict: dict,
                  ckpt_path: str = '../models/mlm/wandb/run-20251128_073215-roformer_13218339_1024_128_15_maskprob_12_5overlap/files/ckpt/epoch=65-step=665082.ckpt',
                  embedder_model: str = 'roformer',
                  chunking_strategy:str='overlap',
                  seq_length_q: int =1024,
                  overlap_q: int=0,
                  seq_length_h: int =256,
                  overlap_h: int= 0,
                  window_hours: float= 6.0
                 )-> None:
    
    assert chunking_strategy in ['overlap','time','visit','care_stage']
    
    seq_gen_q = SequencesGenerator(tokenizer_path=tokenizer_path,
                                   chunk_length=seq_length_q,
                                   overlap=overlap_q)
    
    seq_gen_h = SequencesGenerator(tokenizer_path=tokenizer_path,
                                   chunk_length=seq_length_h,
                                   overlap=overlap_h)
    
    
    data_idx = pl.read_parquet(data_idx_path)
    hf_dataset= load_from_disk(hf_dataset_path)
    
    ConfigClassRet, ModelClassRet = get_config_and_model_cls(model_type=embedder_model, mode='eval', variant=None)

    cfg_ret = ConfigClassRet(vocab_size=seq_gen_q.tokenizer.vocab_size,
                             hidden_size=768,
                             num_hidden_layers=12,
                             num_attention_heads=12,
                             intermediate_size=3072,
                             max_position_embeddings=1536,
                             pad_token_id=0,
                             type_vocab_size= 28,
                             visit_vocab_size= 102,
                             stage_vocab_size= 5)

    embedder = EHREmbedder(
        config=cfg_ret,
        backbone=ModelClassRet,
        ckpt_path=ckpt_path,
        pooling='mean')
    
    
    if chunking_strategy == 'overlap':
        needed_cols = ['subject_id','input_ids','attention_mask','visit_ids','stage_ids','type_ids']
    elif chunking_strategy == 'time':
        needed_cols = ['subject_id','input_ids','attention_mask','visit_ids','stage_ids','type_ids','time_stamp']
    elif chunking_strategy == 'visit':
        needed_cols = ['subject_id','input_ids','attention_mask','visit_ids','stage_ids','type_ids','seq_id']
    elif chunking_strategy == 'care_stage':
        needed_cols = ['subject_id','input_ids','attention_mask','visit_ids','stage_ids','type_ids',
                       'seq_id', 'out_id', 'er_id', 'hadm_id', 'icustay_id']
    
    sub_ids = set(data_idx.get_column("subject_id").to_list())
    hf_dataset = hf_dataset.filter(lambda sids: [sid in sub_ids for sid in sids],
                                   batched=True,
                                   input_columns="subject_id")

    hf_dataset = (hf_dataset
                       .flatten_indices()
                       .select_columns(needed_cols)
                       .with_format("numpy", columns=needed_cols, 
                                    output_all_columns=False))

    index =  _build_index(hf_dataset)
    
    for i in tqdm(range(len(data_idx))):
        row_idx = i
        stay = data_idx[row_idx]

        subject_id = stay['subject_id'][0]
        stay_id = stay['icustay_id'][0]

        timeline_encoded = hf_dataset.select(index[subject_id])[0]

        start_limit_q = limits_dict[main_window_q][seq_length_q][0]
        end_limit_q = limits_dict[main_window_q][seq_length_q][1]
        start_q = stay[start_limit_q][0]
        end_q = stay[end_limit_q][0]


        query = {k: _to_list(v) for k, v in timeline_encoded.items()}
        query = {k: (v[start_q:end_q] if isinstance(v, list) else v) for k, v in query.items()}
        query = seq_gen_q.get_overlapped_chunks(timeline=query,add_cls_per_chunk=True)[0]
        query = {k: query[k] for k in ["input_ids", "attention_mask", "visit_ids", "stage_ids", "type_ids"] if k in query}
        query = {k: torch.tensor(v) for k, v in query.items()}


        start_limit_h = limits_dict[main_window_h][seq_length_q][0]
        end_limit_h = limits_dict[main_window_h][seq_length_q][1]
        start_h = stay[start_limit_h][0] if isinstance(start_limit_h, str) else int(start_limit_h)
        end_h = stay[end_limit_h][0] if isinstance(end_limit_h, str) else int(end_limit_h)

        
        history = {k: _to_list(v) for k, v in timeline_encoded.items()} 
        history = {k: (v[start_h:end_h] if isinstance(v, list) else v) for k, v in history.items()}
        
        if chunking_strategy == 'overlap':
            history = seq_gen_h.get_overlapped_chunks(timeline=history,
                                                      add_cls_per_chunk=True)
        elif chunking_strategy == 'time':
            history = seq_gen_h.get_time_based_chunks(timeline=history,
                                                     window_hours=window_hours,
                                                     keep_prefix_tokens=True,
                                                     anchor_from_first_valid_time=True)
        elif chunking_strategy == 'visit':
            history = seq_gen_h.get_visit_level_chunks(timeline=history,
                                                      keep_prefix_tokens=True)
        elif chunking_strategy == 'care_stage':
            history = seq_gen_h.get_care_stage_level_chunks(timeline=history,
                                                           keep_prefix_tokens=True)
        history = [{k: chunk[k] for k in ["input_ids", "attention_mask", "visit_ids", "stage_ids", "type_ids"] 
                    if k in chunk} for chunk in history]
        history = [{k: torch.tensor(v) for k, v in chunk.items()} for chunk in history]
        history = {k: torch.stack([c[k] for c in history], dim=0) for k in history[0].keys()}

        q_emb = embedder.encode(input_ids=query['input_ids'].unsqueeze(0),
                                attention_mask=query['attention_mask'].unsqueeze(0),
                                type_ids=query['type_ids'].unsqueeze(0),
                                visit_ids=query['visit_ids'].unsqueeze(0),
                                stage_ids=query['stage_ids'].unsqueeze(0))

        ch_emb = embedder.encode(input_ids=history['input_ids'],
                                 attention_mask=history['attention_mask'],
                                 type_ids=history['type_ids'],
                                 visit_ids=history['visit_ids'],
                                 stage_ids=history['stage_ids'])

        ch_emb = torch.cat((ch_emb,q_emb),dim=0)

        idx, _ = build_faiss_index(ch_embs=ch_emb, metric='cosine')
        faiss.write_index(idx, os.path.join(save_path,f"{stay_id}.faiss"))

In [15]:
# build_indices(data_idx_path='../downstream_idx.parquet',
#               hf_dataset_path='../data/meds_normalized_arrow/',
#               tokenizer_path='../vocab.json',
#               save_path='./dummy/',
#               main_window_q='within24_query',
#               main_window_h='within24_hist_full',
#               limits_dict=limits,
#               ckpt_path='../models/mlm/wandb/run-20251128_073215-roformer_13218339_1024_128_15_maskprob_12_5overlap/files/ckpt/epoch=65-step=665082.ckpt',
#               embedder_model='roformer',
#               chunking_strategy='care_stage',
#               seq_length_q=1024,
#               overlap_q=0,
#               seq_length_h=256,
#               overlap_h=0,
#               window_hours=6.0
#               )

In [16]:
import os
import json
import torch
import faiss
import bisect
import chromadb
import numpy as np
import polars as pl

from torch import nn

from math import ceil
from pathlib import Path
from collections import defaultdict
from datasets import load_from_disk
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from typing import Dict, Iterable, List, Optional, Any, Union, Literal, Tuple, Callable

In [17]:
class RetrievalDataset(Dataset):
    def __init__(self,
                 dataset_path: str,
                 data_idx_path: str,
                 vectordb_path:str,
                 tokenizer_path:str,
                 limits_dict: dict,
                 chunking_strategy:str = 'overlap',
                 task: str = 'y_mort',
                 query_window: str = 'within48_query',
                 history_window: str = 'within48_hist_full', 
                 top_k: int = 8,
                 seq_length_q: int = 512,
                 overlap_q: int= 0,
                 seq_length_h: int = 256,
                 overlap_h: int = 0,
                 use_time: bool = True,
                 use_numeric: bool = False,
                 add_cls=True,
                 window_hours: float = 6.0,
                 split: str = 'train') -> None:
        
        assert chunking_strategy in ['overlap','time','visit','care_stage']
        
        self.chunking_strategy = chunking_strategy
        
        if self.chunking_strategy == 'overlap':
            needed_cols = ['subject_id','input_ids','attention_mask','visit_ids','stage_ids','type_ids']
        elif self.chunking_strategy == 'time':
            needed_cols = ['subject_id','input_ids','attention_mask','visit_ids','stage_ids','type_ids','time_stamp']
        elif self.chunking_strategy == 'visit':
            needed_cols = ['subject_id','input_ids','attention_mask','visit_ids','stage_ids','type_ids','seq_id']
        elif self.chunking_strategy == 'care_stage':
            needed_cols = ['subject_id','input_ids','attention_mask','visit_ids','stage_ids','type_ids',
                           'seq_id', 'out_id', 'er_id', 'hadm_id', 'icustay_id']
        
        
        
        if use_time:
            needed_cols.append('time_diff')
        if use_numeric:
            needed_cols.append('numeric_values')
            needed_cols.append('numeric_mask')
            
            
        self.start_limit_q = limits_dict[query_window][seq_length_q][0]
        self.end_limit_q   = limits_dict[query_window][seq_length_q][1]
        
        self.start_limit_h = limits_dict[history_window][seq_length_q][0]
        self.end_limit_h   = limits_dict[history_window][seq_length_q][1]
        
        
        self.task = task
        self.add_cls = add_cls
        
        self.query_gen = SequencesGenerator(tokenizer_path=tokenizer_path,
                                            chunk_length=seq_length_q,
                                            overlap=overlap_q)
        self.history_gen = SequencesGenerator(tokenizer_path=tokenizer_path,
                                            chunk_length=seq_length_h,
                                            overlap=overlap_h)
        
        
        self.vectordb_path = vectordb_path
        self.top_k = top_k
        self.window_hours = window_hours
        
        self.data_idx =  pl.scan_parquet(data_idx_path).collect()
        self.data_idx =  self.data_idx.filter(pl.col('split') == split)
        
        sub_ids = set(self.data_idx.get_column("subject_id").to_list())
        hf_dataset = load_from_disk(dataset_path)

        
        hf_dataset = hf_dataset.filter(
            lambda sids: [sid in sub_ids for sid in sids],
            batched=True,
            input_columns="subject_id",)

        self.hf_dataset = (
            hf_dataset
            .flatten_indices()
            .select_columns(needed_cols)
            .with_format("numpy", columns=needed_cols, output_all_columns=False))
        
        
        sids = self.hf_dataset["subject_id"]        
        self.index = defaultdict(list)
        for i, sid in enumerate(sids):
            self.index[sid].append(i)
        

        
    def __len__(self) -> int:
        return len(self.data_idx)
    

    def __getitem__(self,
                    idx: int):
        
        stay = self.data_idx[idx]
        subject_id = stay['subject_id'][0]
        stay_id = stay['icustay_id'][0]
        label = stay[self.task][0]
        
        timeline_encoded = self.hf_dataset.select(self.index[subject_id])[0]
        
        start_q = stay[self.start_limit_q][0]
        end_q = stay[self.end_limit_q][0]
        start_h = stay[self.start_limit_h][0] if isinstance(self.start_limit_h, str) else int(self.start_limit_h)
        end_h = stay[self.end_limit_h][0] if isinstance(self.end_limit_h, str) else int(self.end_limit_h)


        query = {k: (v[start_q:end_q] if isinstance(v, (list, np.ndarray)) else v) for k, v in timeline_encoded.items()}
        query = self.query_gen.get_overlapped_chunks(query, add_cls_per_chunk=self.add_cls)[0]
        
        history = {k: (v[start_h:end_h] if isinstance(v, (list, np.ndarray)) else v) for k, v in timeline_encoded.items()}
        
        if self.chunking_strategy == 'overlap':
            history = self.history_gen.get_overlapped_chunks(history, 
                                                             add_cls_per_chunk=self.add_cls)
        elif self.chunking_strategy == 'time':
            history = self.history_gen.get_time_based_chunks(timeline=history,
                                                             window_hours=self.window_hours,
                                                             keep_prefix_tokens=True,
                                                             anchor_from_first_valid_time=True)
        elif self.chunking_strategy == 'visit':
            history = self.history_gen.get_visit_level_chunks(timeline=history,
                                                              keep_prefix_tokens=True)
        elif self.chunking_strategy == 'care_stage':
            history = self.history_gen.get_care_stage_level_chunks(timeline=history,
                                                                   keep_prefix_tokens=True)
        allowed_keys = ["input_ids", "attention_mask", "visit_ids", "stage_ids","type_ids",
                        "time_diff", "numeric_values", "numeric_mask"]
        history = [{k: chunk[k] for k in allowed_keys if k in chunk} for chunk in history]
        
        
        history_idx = faiss.read_index(os.path.join(self.vectordb_path,f'{stay_id}.faiss'))
        qid = history_idx.ntotal-1
        query_embed = history_idx.reconstruct(qid)
        
        sim, ids = self._query_faiss(index=history_idx,q_emb=query_embed, top_k=self.top_k, metric='cosine')

        ids = ids[0].tolist()
        ids = [i for i in ids if (i != -1) and (i != qid) and (0 <= i < len(history))]
        
        history = [history[i] for i in ids]
        
        out = {'query': query,
               'history': history,
               'label':label}
        return out
    

    def _query_faiss(self,index, q_emb, top_k: int = 10, metric: str = "cosine"):

        xq = q_emb
        if xq.ndim == 1:
            xq = xq.reshape(1, -1)
        elif xq.ndim == 2:
            pass
        else:
            raise ValueError(f"q_emb must be (D,) or (B,D). Got {xq.shape}")

        xq = xq.astype(np.float32, copy=False)
        xq = np.ascontiguousarray(xq)

        metric = metric.lower()

        if metric in ("cosine", "ip", "inner_product", "dot"):
#             faiss.normalize_L2(xq)
            scores, ids = index.search(xq, top_k)
            return scores, ids

        elif metric in ("l2", "euclidean"):
            dists, ids = index.search(xq, top_k)
            return dists, ids

        else:
            raise ValueError(f"Unknown metric={metric}. Use 'l2' or 'cosine'.")

In [20]:
ds = RetrievalDataset(data_idx_path='../downstream_idx.parquet',
                      dataset_path= '../data/meds_normalized_arrow/',
                      vectordb_path='/faiss/256/w48/',
                      tokenizer_path='../vocab.json',
                      limits_dict=limits,
                      chunking_strategy = 'overlap',
                      task= 'y_mort',
                      query_window= 'within48_query',
                      history_window= 'within48_hist_full',
                      top_k = 8,
                      seq_length_q = 1024,
                      overlap_q= 0,
                      seq_length_h= 256,
                      overlap_h= 32,
                      use_time= True,
                      use_numeric= True,
                      add_cls=True,
                      window_hours= 6.0,
                      split= 'val')

Loading dataset from disk:   0%|          | 0/70 [00:00<?, ?it/s]

Filter:   0%|          | 0/208980 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/4984 [00:00<?, ? examples/s]

In [2]:
from datasets import load_from_disk

In [2]:
ds = load_from_disk('../data/meds_normalized_arrow/')

Loading dataset from disk:   0%|          | 0/70 [00:00<?, ?it/s]

In [3]:
from datasets import load_from_disk

In [4]:
das = load_from_disk('../data/llm-dataset/')

Loading dataset from disk:   0%|          | 0/402 [00:00<?, ?it/s]

In [8]:
das[0]['within_stay_genhpf']

['event type: icu chart, category: neurological, label: behavior, value: , unit: ',
 'event type: icu chart, category: neurological, label: behavior, value: , unit: ',
 'event type: icu chart, category: neurological, label: behavior, value: , unit: ',
 'event type: icu chart, category: neurological, label: behavior, value: , unit: ',
 'event type: icu chart, category: neurological, label: behavior, value: , unit: ',
 'event type: icu chart, category: neurological, label: gcs - motor response, value: 5.0, unit: ',
 'event type: icu chart, category: neurological, label: speech, value: , unit: ',
 'event type: icu chart, category: neurological, label: communication, value: , unit: ',
 'event type: icu chart, category: neurological, label: gag reflex, value: , unit: ',
 'event type: icu chart, category: neurological, label: cough reflex, value: , unit: ',
 'event type: icu chart, category: neurological, label: pupil size right, value: , unit: ',
 'event type: icu chart, category: neurologi